## Setup the data

In [3]:
import os
import time
import numpy as np
import torch
import onnx
import onnxruntime as ort
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Export decoder to ONNX

In [6]:
!python MobileSAM/scripts/export_onnx_model.py \
  --checkpoint immich-sticker-gen/serving/models/distilled_mobile_sam_full.pth \
  --model-type vit_t \
  --output immich-sticker-gen/serving/models/distilled_mobile_sam_decoder.onnx \
  --return-single-mask

/opt/conda/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/opt/conda/lib/python3.12/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/opt/conda/lib/python3.12/site-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_5m_224 in registry with mobile_sam.modeling.tiny_vit_sam.tiny_vit_5m_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  return register_model(fn_wrapper)
/opt/conda/lib/python3.12/site-packages/mobile_sam/modeling/tiny_vit_sam.py:656: UserWarning: Overwriting tiny_vit_11m

## Convert encoder

In [9]:
import torch
import onnx
from mobile_sam import sam_model_registry

device = torch.device("cpu")

base_ckpt = "immich-sticker-gen/serving/models/mobile_sam.pt"
model_path = "immich-sticker-gen/serving/models/distilled_mobile_sam_full.pth"

model = sam_model_registry["vit_t"](checkpoint=base_ckpt)
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

encoder = model.image_encoder
encoder.eval()

encoder_onnx_path = "immich-sticker-gen/serving/models/distilled_mobile_sam_encoder.onnx"
dummy_input = torch.randn(1, 3, 1024, 1024)

torch.onnx.export(
    encoder,
    dummy_input,
    encoder_onnx_path,
    export_params=True,
    opset_version=20,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["image_embeddings"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "image_embeddings": {0: "batch_size"},
    },
)

onnx_model = onnx.load(encoder_onnx_path)
onnx.checker.check_model(onnx_model)

print(f"Encoder ONNX saved to {encoder_onnx_path}")

/opt/conda/lib/python3.12/site-packages/mobile_sam/build_sam.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)
/tmp/ipykernel_10663/1396771679.

Encoder ONNX saved to immich-sticker-gen/serving/models/distilled_mobile_sam_encoder.onnx


## Load test data from SA-1B

In [7]:
SHARD_URL = "https://scontent.xx.fbcdn.net/m1/v/t6/An_YmP5OIPXun-vu3hkckAZZ2s4lPYoVkiyvCcWiVY21mu1Ng5_1HeCa2CWiSTsskj8HQ8bN013HxNpYDdSC_7jWQq_svcg.tar?_nc_gid&ccb=10-5&oh=00_Af08ZycNdYdNMlMCW5sVXhQ0W2iYlA4GO1vtsjm6IY-yYw&oe=69F57028&_nc_sid=0fdd51"

import os
import json
import requests
import tarfile
from pathlib import Path
from PIL import Image

def prepare_sa1b_subset(
    shard_url,
    out_dir="benchmark_sa1b",
    max_images=50,
    tar_path="sa1b_shard.tar",
):
    out_dir = Path(out_dir)
    images_dir = out_dir / "images"
    anns_dir = out_dir / "annotations"
    images_dir.mkdir(parents=True, exist_ok=True)
    anns_dir.mkdir(parents=True, exist_ok=True)

    # download once
    if not os.path.exists(tar_path):
        print("Downloading shard...")
        with requests.get(shard_url, stream=True) as r:
            r.raise_for_status()
            with open(tar_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print("Download complete.")
    else:
        print("Shard already exists, skipping download.")

    manifest_path = out_dir / "manifest.json"
    if manifest_path.exists():
        with open(manifest_path, "r") as f:
            manifest = json.load(f)
        if len(manifest) >= max_images:
            print(f"Subset already exists ({len(manifest)} pairs), skipping extraction.")
            return manifest[:max_images]

    print("Extracting image/json pairs...")
    saved = []
    pending_json = {}

    with tarfile.open(tar_path) as tar:
        members = tar.getmembers()

        # first pass: read jsons into memory by stem
        for member in members:
            if member.isfile() and member.name.endswith(".json"):
                stem = Path(member.name).stem
                f = tar.extractfile(member)
                if f is not None:
                    pending_json[stem] = f.read()

        count = 0

        # second pass: extract matching jpg + json
        for member in members:
            if not (member.isfile() and member.name.endswith(".jpg")):
                continue

            stem = Path(member.name).stem
            if stem not in pending_json:
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            img = Image.open(f).convert("RGB")

            img_path = images_dir / f"{stem}.jpg"
            ann_path = anns_dir / f"{stem}.json"

            img.save(img_path, quality=95)

            with open(ann_path, "wb") as jf:
                jf.write(pending_json[stem])

            saved.append({
                "id": stem,
                "image_path": str(img_path),
                "annotation_path": str(ann_path),
            })

            count += 1
            if count >= max_images:
                break

    with open(manifest_path, "w") as f:
        json.dump(saved, f, indent=2)

    print(f"Saved {len(saved)} image/json pairs to {out_dir}")
    return saved

pairs = prepare_sa1b_subset(
    shard_url=SHARD_URL,
    out_dir="benchmark_sa1b",
    max_images=50,
    tar_path="sa1b_shard.tar",
)

Shard already exists, skipping download.
Subset already exists (50 pairs), skipping extraction.


## Calculate metrics

In [10]:
import os
import json
import time
import cv2
import numpy as np
import onnxruntime as ort
from PIL import Image
from pycocotools import mask as mask_utils

# --------------------------------------------------
# config
# --------------------------------------------------
encoder_onnx_path = "immich-sticker-gen/serving/models/distilled_mobile_sam_encoder.onnx"
decoder_onnx_path = "immich-sticker-gen/serving/models/distilled_mobile_sam_decoder.onnx"

num_eval_samples = 50
num_trials = 100
num_throughput_trials = 100

# assumes:
# pairs = prepare_sa1b_subset(...)
# each item has:
# {"id": ..., "image_path": ..., "annotation_path": ...}

# --------------------------------------------------
# sessions
# --------------------------------------------------
sess_opts = ort.SessionOptions()
sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL

enc_sess = ort.InferenceSession(
    encoder_onnx_path,
    sess_options=sess_opts,
    providers=["CPUExecutionProvider"],
)

dec_sess = ort.InferenceSession(
    decoder_onnx_path,
    sess_options=sess_opts,
    providers=["CPUExecutionProvider"],
)

enc_input_name = enc_sess.get_inputs()[0].name

print("Encoder input:", enc_input_name, enc_sess.get_inputs()[0].shape)
print("Decoder inputs:", [x.name for x in dec_sess.get_inputs()])

# --------------------------------------------------
# helpers
# --------------------------------------------------
PIXEL_MEAN = np.array([123.675, 116.28, 103.53], dtype=np.float32)
PIXEL_STD = np.array([58.395, 57.12, 57.375], dtype=np.float32)

def load_pair(pair):
    image = np.array(Image.open(pair["image_path"]).convert("RGB"))
    with open(pair["annotation_path"], "r") as f:
        ann_data = json.load(f)
    return image, ann_data

def preprocess_sam(image_rgb, image_size=1024):
    h, w = image_rgb.shape[:2]
    scale = image_size / max(h, w)
    new_h = int(h * scale + 0.5)
    new_w = int(w * scale + 0.5)

    resized = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    padded = np.zeros((image_size, image_size, 3), dtype=np.float32)
    padded[:new_h, :new_w, :] = resized.astype(np.float32)

    padded = (padded - PIXEL_MEAN) / PIXEL_STD
    x = np.transpose(padded, (2, 0, 1)).astype(np.float32)  # CHW
    return x, (h, w)

def ann_to_mask(ann):
    seg = ann["segmentation"]
    if isinstance(seg["counts"], list):
        rle = mask_utils.frPyObjects(seg, seg["size"][0], seg["size"][1])
    else:
        rle = seg
    mask = mask_utils.decode(rle)
    if mask.ndim == 3:
        mask = mask[..., 0]
    return mask.astype(bool)

def mask_to_box(mask):
    ys, xs = np.where(mask)
    return np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=np.float32)

def compute_iou(pred_mask, gt_mask):
    inter = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return inter / union if union > 0 else 0.0

def resize_box_to_1024(box_xyxy, original_size, target_length=1024):
    h, w = original_size
    scale = target_length / max(h, w)
    return box_xyxy * scale

def run_onnx_box_prompt(image, box_xyxy):
    x, orig_size = preprocess_sam(image)
    x = np.ascontiguousarray(x[None, ...], dtype=np.float32)  # batch=1 only

    image_embeddings = enc_sess.run(None, {enc_input_name: x})[0]

    scaled_box = resize_box_to_1024(box_xyxy.copy(), orig_size)

    # SAM box prompt format: two points with labels 2 and 3
    point_coords = np.array(
        [[[scaled_box[0], scaled_box[1]], [scaled_box[2], scaled_box[3]]]],
        dtype=np.float32,
    )
    point_labels = np.array([[2, 3]], dtype=np.float32)

    mask_input = np.zeros((1, 1, 256, 256), dtype=np.float32)
    has_mask_input = np.zeros((1,), dtype=np.float32)
    orig_im_size = np.array([orig_size[0], orig_size[1]], dtype=np.float32)

    ort_inputs = {
        "image_embeddings": image_embeddings,
        "point_coords": point_coords,
        "point_labels": point_labels,
        "mask_input": mask_input,
        "has_mask_input": has_mask_input,
        "orig_im_size": orig_im_size,
    }

    masks, scores, low_res_masks = dec_sess.run(None, ort_inputs)
    pred_mask = masks[0, 0] > 0.0
    return pred_mask, scores

# --------------------------------------------------
# subset
# --------------------------------------------------
eval_pairs = pairs[:min(num_eval_samples, len(pairs))]

# --------------------------------------------------
# accuracy = Mean IoU
# --------------------------------------------------
ious = []

for pair in eval_pairs:
    image, ann_data = load_pair(pair)
    anns = ann_data.get("annotations", [])
    if not anns:
        continue

    gt_mask = ann_to_mask(anns[0])
    box = mask_to_box(gt_mask)

    pred_mask, _ = run_onnx_box_prompt(image, box)
    ious.append(compute_iou(pred_mask, gt_mask))

mean_iou = 100.0 * np.mean(ious) if ious else 0.0

# --------------------------------------------------
# single-sample latency (full ONNX pipeline)
# --------------------------------------------------
single_image, single_ann_data = load_pair(eval_pairs[0])
single_gt_mask = ann_to_mask(single_ann_data["annotations"][0])
single_box = mask_to_box(single_gt_mask)

_ = run_onnx_box_prompt(single_image, single_box)

latencies = []
for _ in range(num_trials):
    t0 = time.time()
    _ = run_onnx_box_prompt(single_image, single_box)
    latencies.append(time.time() - t0)

# --------------------------------------------------
# throughput: repeated batch=1 encoder runs
# --------------------------------------------------
encoder_times = []
images_processed = 0

# warmup
warm_x, _ = preprocess_sam(single_image)
warm_x = np.ascontiguousarray(warm_x[None, ...], dtype=np.float32)
_ = enc_sess.run(None, {enc_input_name: warm_x})

for _ in range(num_throughput_trials):
    for pair in eval_pairs:
        image, _ = load_pair(pair)
        x, _ = preprocess_sam(image)
        x = np.ascontiguousarray(x[None, ...], dtype=np.float32)

        t0 = time.time()
        _ = enc_sess.run(None, {enc_input_name: x})
        encoder_times.append(time.time() - t0)
        images_processed += 1

encoder_fps = images_processed / np.sum(encoder_times)

# --------------------------------------------------
# report
# --------------------------------------------------
encoder_size = os.path.getsize(encoder_onnx_path) / 1e6
decoder_size = os.path.getsize(decoder_onnx_path) / 1e6

print(f"Mean IoU: {mean_iou:.2f}% ({len(ious)} samples)")
print(f"Encoder ONNX Size: {encoder_size:.2f} MB")
print(f"Decoder ONNX Size: {decoder_size:.2f} MB")
print(f"Total ONNX Size: {encoder_size + decoder_size:.2f} MB")
print(f"Inference Latency (single sample, median): {np.percentile(latencies, 50) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 95th percentile): {np.percentile(latencies, 95) * 1000:.2f} ms")
print(f"Inference Latency (single sample, 99th percentile): {np.percentile(latencies, 99) * 1000:.2f} ms")
print(f"Inference Throughput (single sample): {num_trials / np.sum(latencies):.2f} FPS")
print(f"Encoder Throughput (batch=1 repeated): {encoder_fps:.2f} FPS")

Encoder input: input ['batch_size', 3, 1024, 1024]
Decoder inputs: ['image_embeddings', 'point_coords', 'point_labels', 'mask_input', 'has_mask_input', 'orig_im_size']
Mean IoU: 80.60% (50 samples)
Encoder ONNX Size: 28.06 MB
Decoder ONNX Size: 16.51 MB
Total ONNX Size: 44.57 MB
Inference Latency (single sample, median): 301.44 ms
Inference Latency (single sample, 95th percentile): 348.72 ms
Inference Latency (single sample, 99th percentile): 360.94 ms
Inference Throughput (single sample): 3.24 FPS
Encoder Throughput (batch=1 repeated): 4.27 FPS
